# Portfolio 2 — NHANES Dietary Predictors of Glycaemic Status
## Notebook 01: Data Download and Merge

**Goal:** Download NHANES 2017–2018 data files from the CDC, merge them on participant ID (SEQN),
apply exclusion criteria, and save a clean dataset for downstream analysis.

**Files merged:**

| File | Contents |
|---|---|
| DEMO_J.XPT | Demographics: age, sex, race, income, pregnancy status |
| GLU_J.XPT | Fasting glucose + insulin |
| GHB_J.XPT | HbA1c (glycated haemoglobin) |
| DR1TOT_J.XPT | Dietary recall — Day 1 |
| DR2TOT_J.XPT | Dietary recall — Day 2 |
| PAQ_J.XPT | Physical activity questionnaire |
| BMX_J.XPT | Body measurements: BMI, waist circumference |

Data source: [CDC NHANES 2017-2018](https://wwwn.cdc.gov/nchs/nhanes/search/datapage.aspx?Component=Questionnaire&CycleBeginYear=2017)


In [ ]:
import pandas as pd
import os
import warnings
warnings.filterwarnings('ignore', category=pd.errors.PerformanceWarning)

## Step 1: Load each XPT file into a DataFrame

In [2]:
data_dir = '/Users/larascofano/Documents/Obsidian Vault/career/Portfolio 2/data'

demo               = pd.read_sas(f"{data_dir}/DEMO_J.xpt",    format='xport')
glu                = pd.read_sas(f"{data_dir}/GLU_J.xpt",     format='xport')
ghb                = pd.read_sas(f"{data_dir}/GHB_J.xpt",     format='xport')
dietary1           = pd.read_sas(f"{data_dir}/DR1TOT_J.xpt",  format='xport')
dietary2           = pd.read_sas(f"{data_dir}/DR2TOT_J.xpt",  format='xport')
physical_activity  = pd.read_sas(f"{data_dir}/PAQ_J.xpt",     format='xport')
BMI                = pd.read_sas(f"{data_dir}/BMX_J.xpt",     format='xport')
alq                = pd.read_sas(f"{data_dir}/ALQ_J.xpt",     format='xport')
smo                = pd.read_sas(f"{data_dir}/SMQ_J.xpt",     format='xport')

print("Files loaded successfully")

Files loaded successfully


/var/folders/j7/7v8b_5h52qv3cmmqrvz2xkzw0000gn/T/ipykernel_27094/3536587066.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  dietary1           = pd.read_sas(f"{data_dir}/DR1TOT_J.xpt",  format='xport')
/var/folders/j7/7v8b_5h52qv3cmmqrvz2xkzw0000gn/T/ipykernel_27094/3536587066.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  dietary1           = pd.read_sas(f"{data_dir}/DR1TOT_J.xpt",  format='xport')
/var/folders/j7/7v8b_5h52qv3cmmqrvz2xkzw0000gn/T/ipykernel_27094/3536587066.py:6: PerformanceWarning: DataFrame is highly 

## Step 2: Quick check on each DataFrame

In [3]:
dataframes = {
    'demo':              demo,
    'glu':               glu,
    'ghb':               ghb,
    'dietary1':          dietary1,
    'dietary2':          dietary2,
    'physical_activity': physical_activity,
    'BMI':               BMI,
    'alq':               alq,
    'smo':               smo,
}

for name, df in dataframes.items():
    print(f"  {name:<20} shape: {df.shape}")

  demo                 shape: (9254, 46)
  glu                  shape: (3036, 4)
  ghb                  shape: (6401, 2)
  dietary1             shape: (8704, 168)
  dietary2             shape: (8704, 85)
  physical_activity    shape: (5856, 17)
  BMI                  shape: (8704, 21)
  alq                  shape: (5533, 10)
  smo                  shape: (6724, 37)


## Step 3: Merge dietary recall days and average macronutrients

Participants completed dietary recall on 2 separate days. Averaging gives a more reliable estimate
of habitual intake than a single day (which has high day-to-day variability).

In [4]:
dietary_merged = dietary1.merge(dietary2, how='left', on='SEQN')

dietary_merged['carb_avg']  = dietary_merged[['DR1TCARB', 'DR2TCARB']].mean(axis=1)
dietary_merged['sugar_avg'] = dietary_merged[['DR1TSUGR', 'DR2TSUGR']].mean(axis=1)
dietary_merged['fiber_avg'] = dietary_merged[['DR1TFIBE', 'DR2TFIBE']].mean(axis=1)
dietary_merged['fat_avg']   = dietary_merged[['DR1TTFAT', 'DR2TTFAT']].mean(axis=1)
dietary_merged['kcal_avg']  = dietary_merged[['DR1TKCAL', 'DR2TKCAL']].mean(axis=1)

print(f"Dietary merged shape: {dietary_merged.shape}")
dietary_merged[['SEQN','carb_avg','sugar_avg','fiber_avg','fat_avg','kcal_avg']].head(3)

Dietary merged shape: (8704, 257)


,SEQN,carb_avg,sugar_avg,fiber_avg,fat_avg,kcal_avg
0,93703.0,NaN,NaN,NaN,NaN,NaN
1,93704.0,195.075,105.670,7.05,40.130,1293.0
2,93705.0,152.470,67.295,10.75,56.085,1218.5


## Step 4: Select columns of interest and rename to readable names

In [5]:
# Select only the variables we need
demographics = demo[['SEQN', 'RIDAGEYR', 'RIAGENDR', 'RIDRETH3', 'INDFMPIR', 'WTMEC2YR', 'RIDEXPRG']]
glu_outcome  = glu[['SEQN', 'LBXGLU']]
ghb_outcome  = ghb[['SEQN', 'LBXGH']]
diet         = dietary_merged[['SEQN', 'carb_avg', 'sugar_avg', 'fiber_avg', 'fat_avg', 'kcal_avg']]
activity     = physical_activity[['SEQN', 'PAQ650', 'PAQ665']]
BMI_cols     = BMI[['SEQN', 'BMXBMI', 'BMXWAIST']]

# Rename NHANES codes to readable names
demographics = demographics.rename(columns={
    'RIDAGEYR': 'age', 'RIAGENDR': 'sex', 'RIDRETH3': 'race',
    'INDFMPIR': 'income', 'WTMEC2YR': 'survey_weight', 'RIDEXPRG': 'is_pregnant'
})
glucose  = glu_outcome.rename(columns={'LBXGLU': 'fasting glucose (mg/DL)'})
ghb      = ghb_outcome.rename(columns={'LBXGH': 'HbA1c (%)'})
activity = activity.rename(columns={'PAQ650': 'vigorous activity', 'PAQ665': 'moderate activity'})
body     = BMI_cols.rename(columns={'BMXBMI': 'BMI', 'BMXWAIST': 'waist circumference'})

print("Columns renamed")

Columns renamed


## Step 5: Merge all files on SEQN (participant ID)

In [6]:
df = (demographics
      .merge(glucose,  on='SEQN', how='left')
      .merge(diet,     on='SEQN', how='left')
      .merge(ghb,      on='SEQN', how='left')
      .merge(activity, on='SEQN', how='left')
      .merge(body,     on='SEQN', how='left'))

print(f"Merged shape: {df.shape}")
df.head(2)

Merged shape: (9254, 18)


,SEQN,age,sex,race,income,survey_weight,is_pregnant,fasting glucose (mg/DL),carb_avg,sugar_avg,fiber_avg,fat_avg,kcal_avg,HbA1c (%),vigorous activity,moderate activity,BMI,waist circumference
0,93703.0,2.0,2.0,6.0,5.0,8539.731348,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,17.5,48.2
1,93704.0,2.0,1.0,3.0,5.0,42566.614750,NaN,NaN,195.075,105.67,7.05,40.13,1293.0,NaN,NaN,NaN,15.7,50.0


## Step 6: Data quality check — missing values before exclusions

In [7]:
print(f"Total participants before exclusions: {len(df)}")
print()
missing_pct = (df.isnull().sum() / len(df) * 100).round(1)
print("Missing values (%):")
print(missing_pct[missing_pct > 0].to_string())

Total participants before exclusions: 9254

Missing values (%):
income                     13.3
is_pregnant                88.0
fasting glucose (mg/DL)    68.8
carb_avg                   19.0
sugar_avg                  19.0
fiber_avg                  19.0
fat_avg                    19.0
kcal_avg                   19.0
HbA1c (%)                  34.7
vigorous activity          36.7
moderate activity          36.7
BMI                        13.5
waist circumference        17.9


## Step 7: Apply exclusion criteria

| Criterion | Reason |
|---|---|
| Pregnant (is_pregnant == 1) | Different glucose physiology during pregnancy |
| Missing fasting glucose | Primary outcome unavailable |
| Age < 18 | Adult population only |


In [8]:
print(f"Before exclusions:          {len(df):>5}")

df = df[~(df['is_pregnant'] == 1)]
print(f"After excluding pregnant:   {len(df):>5}")

df = df[df['fasting glucose (mg/DL)'].notna()]
print(f"After excluding missing glu:{len(df):>5}")

df = df[df['age'] >= 18]
print(f"After excluding age < 18:   {len(df):>5}")

Before exclusions:           9254
After excluding pregnant:    9199
After excluding missing glu: 2871
After excluding age < 18:    2517


## Step 8: Save clean dataset

In [9]:
df = df.round(2)
output_path = '/Users/larascofano/Documents/Obsidian Vault/career/Portfolio 2/data/nhanes_clean.csv'
df.to_csv(output_path, index=False)

print(f"Saved: {output_path}")
print(f"Final shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")

Saved: /Users/larascofano/Documents/Obsidian Vault/career/Portfolio 2/data/nhanes_clean.csv
Final shape: (2517, 18)
Columns: ['SEQN', 'age', 'sex', 'race', 'income', 'survey_weight', 'is_pregnant', 'fasting glucose (mg/DL)', 'carb_avg', 'sugar_avg', 'fiber_avg', 'fat_avg', 'kcal_avg', 'HbA1c (%)', 'vigorous activity', 'moderate activity', 'BMI', 'waist circumference']
